In [1]:
import polars as pl
import numpy as np
from scipy import stats
from pathlib import Path

In [2]:
DONOR = 'donor_3'

In [3]:
WORKING_PATH = Path('/group/pmc021/amunif/epi-thesis/workflow/16_Pairwise Ranking Healthy Liver/')
DATASET_PATH = WORKING_PATH / 'dataset'/ DONOR
OUTPUT_PATH  = WORKING_PATH / 'output' / 'combined' / DONOR

In [10]:
# One sample test
def one_sample_test_against_chance(accuracies, p_null=0.5):
    accuracies = np.asarray(accuracies, dtype=float)
    n = len(accuracies)
    mean = accuracies.mean()
    sd = accuracies.std(ddof=1)
    se = sd / np.sqrt(n)
 
    # one-sample t-test -- preferred for small n (e.g. n=5 runs)
    t_stat, t_p = stats.ttest_1samp(accuracies, popmean=p_null)
 
    # z-test shown for reference; not appropriate for n this small
    z_stat = (mean - p_null) / se
    z_p = 2 * (1 - stats.norm.cdf(abs(z_stat)))
 
    return {
        "n_runs": n,
        "mean_accuracy": mean,
        "sd": sd,
        "t_stat": t_stat,
        "t_p_value": t_p,
        "z_stat": z_stat,
        "z_p_value": z_p,
    }

In [11]:
# Pretty report
def report(name, accuracies):
    r = one_sample_test_against_chance(accuracies)
    print(f"{name}:")
    print(f"  mean = {r['mean_accuracy']*100:.2f}% +/- {r['sd']*100:.2f}% "
          f"(n={r['n_runs']} runs)")
    print(f"  t-test:  t({r['n_runs']-1}) = {r['t_stat']:.3f}, "
          f"p = {r['t_p_value']:.4g}")
    print(f"  z-test:  z = {r['z_stat']:.3f}, p = {r['z_p_value']:.4g}")
    print()

In [4]:
### Merge the results into single CSV file

# Read all CSV into single dataframe
pl_df = pl.read_csv(OUTPUT_PATH / 'test' / "*-test-metrics.csv")

In [5]:
pl_df

model,seed,histone_marker,epochs_trained,val_accuracy,val_auc,test_accuracy,test_auc,test_precision,test_recall,test_f1,antisymmetry
str,i64,str,i64,f64,f64,f64,f64,f64,f64,f64,f64
"""DirectRanker""",1011,"""H3K4me3""",84,71.4,0.79,71.9,0.8161,0.7035,0.761,0.7311,0.894
"""LogisticRegression""",1011,"""H3K4me3""",null,70.0,0.7628,71.0,0.7955,0.75,0.6335,0.6868,0.793
"""RandomForest""",1011,"""H3K4me3""",null,70.2,0.7707,73.1,0.8097,0.7716,0.6594,0.7111,0.775
"""SVM_Linear""",1011,"""H3K4me3""",null,70.1,0.7646,71.3,0.7959,0.7541,0.6355,0.6897,0.794
"""DirectRanker""",123,"""H3K4me3""",60,74.8,0.84,71.7,0.8055,0.6917,0.7663,0.7271,0.899
…,…,…,…,…,…,…,…,…,…,…,…
"""SVM_Linear""",456,"""H3K4me3-H3K9ac-H3K9me3-H3K27ac…",null,72.1,0.7833,70.9,0.7483,0.7231,0.6502,0.6847,0.757
"""DirectRanker""",789,"""H3K4me3-H3K9ac-H3K9me3-H3K27ac…",67,73.4,0.81,74.6,0.8397,0.7371,0.7694,0.7529,0.96
"""LogisticRegression""",789,"""H3K4me3-H3K9ac-H3K9me3-H3K27ac…",null,71.9,0.7735,71.6,0.781,0.7483,0.6561,0.6992,0.761


# Chek for the H3K9me3 results

In [8]:
# Check for the H3K9me3
H3K9me3_DirectRanker_df = pl_df.filter((pl.col("model") == "DirectRanker") & (pl.col("histone_marker") == "H3K9me3"))
H3K9me3_DirectRanker_df

model,seed,histone_marker,epochs_trained,val_accuracy,val_auc,test_accuracy,test_auc,test_precision,test_recall,test_f1,antisymmetry
str,i64,str,i64,f64,f64,f64,f64,f64,f64,f64,f64
"""DirectRanker""",1011,"""H3K9me3""",100,49.1,0.5,51.3,0.5159,0.5078,0.9721,0.6671,0.075
"""DirectRanker""",123,"""H3K9me3""",100,52.2,0.53,49.3,0.5127,0.4922,0.9634,0.6515,0.073
"""DirectRanker""",42,"""H3K9me3""",100,48.2,0.53,49.1,0.525,0.4855,0.9751,0.6482,0.058
"""DirectRanker""",456,"""H3K9me3""",100,48.7,0.52,49.2,0.5104,0.4886,0.9671,0.6492,0.073
"""DirectRanker""",789,"""H3K9me3""",100,48.5,0.51,50.7,0.5127,0.5051,0.9801,0.6667,0.049


In [14]:
# Select the accuracy
test_accuracies = H3K9me3_DirectRanker_df["test_accuracy"].to_list()
test_accuracies = np.array(test_accuracies) / 100
test_accuracies

array([0.513, 0.493, 0.491, 0.492, 0.507])

In [15]:
report("H3K9me3", test_accuracies)

H3K9me3:
  mean = 49.92% +/- 1.01% (n=5 runs)
  t-test:  t(4) = -0.177, p = 0.8681
  z-test:  z = -0.177, p = 0.8595



# Check for H3K27me3

In [21]:
# Check for the H3K27me3
H3K27me3_DirectRanker_df = pl_df.filter((pl.col("model") == "DirectRanker") & (pl.col("histone_marker") == "H3K27me3"))
H3K27me3_DirectRanker_df

model,seed,histone_marker,epochs_trained,val_accuracy,val_auc,test_accuracy,test_auc,test_precision,test_recall,test_f1,antisymmetry
str,i64,str,i64,f64,f64,f64,f64,f64,f64,f64,f64
"""DirectRanker""",1011,"""H3K27me3""",100,54.3,0.6,57.0,0.615,0.5412,0.9422,0.6875,0.255
"""DirectRanker""",123,"""H3K27me3""",100,59.6,0.64,55.8,0.6192,0.5285,0.9431,0.6774,0.264
"""DirectRanker""",42,"""H3K27me3""",100,55.5,0.63,55.3,0.6062,0.5194,0.948,0.6711,0.256
"""DirectRanker""",456,"""H3K27me3""",100,54.7,0.61,55.8,0.6153,0.5255,0.9342,0.6726,0.267
"""DirectRanker""",789,"""H3K27me3""",100,56.0,0.63,55.3,0.5941,0.5317,0.9324,0.6773,0.25


In [18]:
# Select the accuracy
test_accuracies = H3K27me3_DirectRanker_df["test_accuracy"].to_list()
test_accuracies = np.array(test_accuracies) / 100
test_accuracies

array([0.57 , 0.558, 0.553, 0.558, 0.553])

In [19]:
report("H3K27me3", test_accuracies)

H3K27me3:
  mean = 55.84% +/- 0.69% (n=5 runs)
  t-test:  t(4) = 18.790, p = 4.724e-05
  z-test:  z = 18.790, p = 0

